# z612 - LightGBM Cliente-Producto (baseline, hiperparametros fijos)
Aisla el efecto de cambiar la granularidad sola, antes de sumar tuning/walk-forward de nuevo. El submit final SUMA las predicciones por customer_id dentro de cada product_id (Kaggle pide a nivel producto).

In [1]:
!pip install -q lightgbm pyarrow

In [2]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'LGB08_CP602',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': '/home/ds/exp/CP602/',
    'archivo_features': 'tb_features_CP602.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'periodo_target_final': 202002,
    'semilla': 102103
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB08_CP602


## 1. Cargar features y armar target a horizonte 2 (por par cliente-producto)

In [4]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

CLAVE = ["customer_id", "product_id"]

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(CLAVE + ["periodo"])

H = PARAM['horizonte_meses']

df = df.with_columns(
    pl.col("tn").shift(-H).over(CLAVE).alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

## 2. Split train / valid (por periodo TARGET, igual criterio que las etapas anteriores)

In [5]:
m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

df_valido = df.filter(pl.col("tn_target").is_not_null())

train = df_valido.filter(pl.col("periodo_target_m") <= m_201910)
valid = df_valido.filter(
    (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
)

print("train:", train.height, " valid:", valid.height)

train: 14227361  valid: 1036089


## 3. Preparar matrices

In [6]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["customer_id", "product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

train_pd = a_pandas(train)
valid_pd = a_pandas(valid)

X_train = train_pd[features]
y_train = np.log1p(train_pd["tn_target"].clip(lower=0))

X_valid = valid_pd[features]
y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

## 4. Entrenar LightGBM (hiperparametros fijos -- aisla el efecto de la granularidad sola)

In [7]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': PARAM['semilla']
}

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                      params={'feature_pre_filter': False})
dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                      params={'feature_pre_filter': False})

modelo = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.111993	valid's rmse: 0.104692
[200]	train's rmse: 0.109606	valid's rmse: 0.104704
Early stopping, best iteration is:
[136]	train's rmse: 0.110888	valid's rmse: 0.104612
mejor iteracion: 136


## 5. Prediccion para 202002 (por par cliente-producto) y agregacion a product_id
Kaggle pide el submit a nivel producto -- se predice por par y se SUMAN las predicciones de todos los clientes dentro de cada producto.

In [8]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_log = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
pred_tn = np.expm1(pred_log)
pred_tn = np.clip(pred_tn, 0, None)

resultado_cp = futuro.select(["customer_id", "product_id"]).to_pandas()
resultado_cp["tn"] = pred_tn

resultado = resultado_cp.groupby("product_id", as_index=False)["tn"].sum()
print(resultado.shape)

(927, 2)


## 6. Filtrar a productos a predecir y armar submit

In [9]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
/home/ds/exp/LGB08_CP602/LGB08_CP602_submit.csv


,product_id,tn
0,20001,965.841024
1,20002,631.658359
2,20003,541.024576
3,20004,416.652015
4,20005,407.726081


## 7. Submit a Kaggle

In [10]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} granularidad cliente-producto")

100%|██████████| 18.6k/18.6k [00:00<00:00, 53.1kB/s]


93 submissions remaining today.
Successfully submitted to Labo III, 2026 BA

In [11]:
importancia = pl.DataFrame({
    "feature": modelo.feature_name(),
    "importancia": modelo.feature_importance(importance_type="gain")
}).sort("importancia", descending=True)

importancia.write_csv(os.path.join(ruta, "feature_importance.csv"))
print(os.path.join(ruta, "feature_importance.csv"))

/home/ds/exp/LGB08_CP602/feature_importance.csv
